# Neuromodulation research in eLife: data collection

Rola Ke, DATASCI 530 (Fall 2026), Project 1, updated for Project 2

## 1. Project description

For this project I collected data on neuroscience articles from [eLife](https://elifesciences.org), an open-access life sciences journal that publishes its articles under a CC-BY 4.0 license. I collected every Neuroscience article that matches the search term "neuromodulation" and recorded each article's title, publication date, article type, number of authors, number of references, the countries in the author affiliations, number of funders, how old its references are, keywords, study organisms, subject areas, and eLife's assessment ratings. I also saved every author affiliation (institution, city, and country) in a second table. I collected the data on October 29, 2026, so articles published after that date are not included and 2026 is only a partial year.

eLife's website `robots.txt` disallows automated access to its search pages, and its article pages show a browser verification check to scripts. Because of this, I collected the data through eLife's public JSON API (`api.elifesciences.org`), which has the same article records and does not need a key. The API's own `robots.txt` (`Disallow: /`) also asks automated tools not to access it. My instructor, Dr. Alejandro Sánchez Becerra, approved using the API for this project. The scraper waits one second after every request and saves author counts, not author names.

The scraper works in three steps:

1. Search: loop through the pages of API search results for "neuromodulation", 100 results per page.
2. Navigation: for each search result, build the address of the article's own record from its ID (`/articles/{id}` or `/reviewed-preprints/{id}`), request that page, and extract the variables listed below.
3. Dataset: put the records into two tables, one row per article and one row per affiliation, and save them as `elife_neuromodulation_raw.csv` and `elife_affiliations_raw.csv`.

### Restrictions

- Search term: "neuromodulation". eLife matches it anywhere in an article's text, not just in the title or abstract.
- Subject: Neuroscience only.
- Article types: research articles and reviewed preprints (eLife's peer-reviewed preprint format, introduced in 2023). Other formats, such as short reports, reviews, and commentary, are left out.
- One request at a time, with a one-second wait after each. A failed request is retried up to two more times.

### Variables

Articles (`elife_neuromodulation_raw.csv`):

| Variable | Description |
|---|---|
| `article_id` | eLife article number |
| `title` | Article title |
| `article_type` | `research-article` or `reviewed-preprint` |
| `published`, `pub_year` | Publication date and year |
| `n_authors` | Number of authors |
| `n_references` | Number of references |
| `countries` | Countries in the authors' affiliations, separated by `;` |
| `n_funders` | Number of different funders listed |
| `reference_age` | Publication year minus the median year of the article's references |
| `keywords` | Author keywords, separated by `;` |
| `research_organisms` | Study organisms, e.g. Mouse or Human |
| `subjects` | eLife subject areas |
| `assessment_significance`, `assessment_strength` | eLife's assessment ratings, only given under its review model from 2023 on |

Affiliations (`elife_affiliations_raw.csv`): `article_id`, `institution`, `city`, and `country`, with one row per distinct affiliation in each article.

### Changes since Project 1

In Project 1 this notebook also cleaned the data. It replaced two country names while collecting them and dropped every row with a missing value, and neither step was backed by a check. I took both out. This notebook now saves the data exactly as the API returns it, and the cleaning happens in the Project 2 report (`elife_neuromodulation_report.ipynb`), next to the checks that justify each step. Because I re-ran the scraper, the counts are a little different from Project 1 (481 articles collected on October 20, 2026).

## 2. Setup

In [1]:
import html
import math
import re
import statistics
import time
from datetime import datetime, timezone

import pandas as pd
import requests

API = "https://api.elifesciences.org"
QUERY = "neuromodulation"
SUBJECT = "neuroscience"
TYPES = ["research-article", "reviewed-preprint"]
PER_PAGE = 100
DELAY = 1.0
HEADERS = {"User-Agent": "DATASCI530 student project (Emory University)"}


def get_json(url, params=None, tries=3):
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=30)
            time.sleep(DELAY)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == tries:
                raise
            time.sleep(5)

## 3. Searching for articles

In [2]:
collection_date = datetime.now(timezone.utc)
params = {"for": QUERY, "subject[]": SUBJECT, "type[]": TYPES,
          "sort": "date", "order": "asc", "per-page": PER_PAGE}

first_page = get_json(f"{API}/search", {**params, "page": 1})
total = first_page["total"]
n_pages = math.ceil(total / PER_PAGE)
search_results = first_page["items"]
for page in range(2, n_pages + 1):
    search_results += get_json(f"{API}/search", {**params, "page": page})["items"]

print(f"Search run on {collection_date:%B %d, %Y at %H:%M} UTC")
print(f"The API reports {total} matching articles; collected {len(search_results)} results from {n_pages} pages.")
pd.DataFrame(search_results)[["id", "type", "published", "title"]].head()

Search run on October 29, 2026 at 12:45 UTC
The API reports 480 matching articles; collected 480 results from 5 pages.


,id,type,published,title
0,00012,research-article,2013-01-22T00:00:00Z,Changing the responses of cortical neurons fro...
1,02265,research-article,2014-05-14T00:00:00Z,<i>Atoh1</i>-dependent rhombic lip neurons are...
2,02598,research-article,2014-06-11T00:00:00Z,Hidden synaptic differences in a neural circui...
3,02938,research-article,2014-06-10T00:00:00Z,<i>Caenorhabditis elegans</i> male sensory-mot...
4,02395,research-article,2014-08-19T00:00:00Z,Shared mushroom body circuits underlie visual ...


## 4. Visiting each article's record

`affiliations()` returns one row per author affiliation. `extract()` uses it to build the list of countries, so both tables come from the same fields.

In [3]:
def clean(text):
    return re.sub(r"<[^>]+>", "", html.unescape(text)).strip()


def as_text(values):
    return "; ".join(clean(v) for v in values) if values else None


def affiliations(item, record):
    rows = []
    for author in record.get("authors") or []:
        for affiliation in author.get("affiliations", []):
            address = (affiliation.get("address") or {}).get("components", {})
            rows.append({"article_id": item["id"],
                         "institution": as_text(affiliation.get("name")),
                         "city": as_text(address.get("locality")),
                         "country": address.get("country")})
    return rows


def extract(item, record):
    authors = record.get("authors")
    references = record.get("references")
    funding = record.get("funding")
    assessment = record.get("elifeAssessment") or {}
    countries = {a["country"] for a in affiliations(item, record)} - {None}
    ref_years = [int(year) for year in (str(r.get("date", ""))[:4] for r in references or []) if year.isdigit()]
    return {
        "article_id": item["id"],
        "title": clean(record["title"]),
        "article_type": item["type"],
        "published": record.get("published"),
        "n_authors": None if authors is None else sum(a.get("type") in ("person", "group") for a in authors),
        "n_references": None if references is None else len(references),
        "countries": "; ".join(sorted(countries)) or None,
        "n_funders": None if funding is None
                     else len({(award.get("source") or {}).get("name", [""])[0] for award in funding.get("awards", [])}),
        "reference_age": int(record["published"][:4]) - statistics.median(ref_years) if ref_years else None,
        "keywords": as_text(record.get("keywords")),
        "research_organisms": as_text(record.get("researchOrganisms")),
        "subjects": as_text([s["name"] for s in record.get("subjects", [])]),
        "assessment_significance": as_text(assessment.get("significance")),
        "assessment_strength": as_text(assessment.get("strength")),
    }

In [4]:
FIELDS = ["authors", "references", "funding", "keywords", "researchOrganisms", "elifeAssessment"]
rows, affiliation_rows, fields_present, failed = [], [], [], []
for i, item in enumerate(search_results, start=1):
    kind = "articles" if item["type"] == "research-article" else "reviewed-preprints"
    try:
        record = get_json(f"{API}/{kind}/{item['id']}")
    except requests.RequestException as err:
        failed.append((item["id"], str(err)))
        continue
    rows.append(extract(item, record))
    affiliation_rows += affiliations(item, record)
    fields_present.append({"article_type": item["type"], **{field: field in record for field in FIELDS}})
    if i % 50 == 0 or i == len(search_results):
        print(f"{i}/{len(search_results)} articles fetched")

print(f"Done. Failed requests: {len(failed)}")

50/480 articles fetched


100/480 articles fetched


150/480 articles fetched


200/480 articles fetched


250/480 articles fetched


300/480 articles fetched


350/480 articles fetched


400/480 articles fetched


450/480 articles fetched


480/480 articles fetched
Done. Failed requests: 0


## 5. Building the dataset

I save both tables without dropping anything.

In [5]:
df = pd.DataFrame(rows)
published = pd.to_datetime(df["published"], utc=True)
df["published"] = published.dt.strftime("%Y-%m-%d")
df["pub_year"] = published.dt.year
df = df.astype({"n_authors": "Int64", "n_references": "Int64", "n_funders": "Int64"})
affils = pd.DataFrame(affiliation_rows).drop_duplicates().reset_index(drop=True)

df.to_csv("elife_neuromodulation_raw.csv", index=False)
affils.to_csv("elife_affiliations_raw.csv", index=False)
print(f"Articles: {len(df)} rows x {df.shape[1]} variables saved to elife_neuromodulation_raw.csv")
print(f"Affiliations: {len(affils)} rows x {affils.shape[1]} variables saved to elife_affiliations_raw.csv")
df.head()

Articles: 480 rows x 15 variables saved to elife_neuromodulation_raw.csv
Affiliations: 1654 rows x 4 variables saved to elife_affiliations_raw.csv


,article_id,title,article_type,published,n_authors,n_references,countries,n_funders,reference_age,keywords,research_organisms,subjects,assessment_significance,assessment_strength,pub_year
0,00012,Changing the responses of cortical neurons fro...,research-article,2013-01-22,5,71,Germany; Switzerland,3,9.0,synaptic plasticity; STDP; visual cortex; circ...,Rat,Neuroscience,None,None,2013
1,02265,Atoh1-dependent rhombic lip neurons are requir...,research-article,2014-05-14,7,103,United States,3,6.0,breathing; central pattern generator; PreBötzi...,Mouse,Neuroscience,None,None,2014
2,02598,Hidden synaptic differences in a neural circui...,research-article,2014-06-11,3,104,United States,3,8.0,Tritonia diomedea; individual variability; syn...,None,Neuroscience,None,None,2014
3,02938,Caenorhabditis elegans male sensory-motor neur...,research-article,2014-06-10,4,100,United States,1,9.0,G-CaMP; acetylcholine; ejaculation; dopamine; ...,C. elegans,Neuroscience,None,None,2014
4,02395,Shared mushroom body circuits underlie visual ...,research-article,2014-08-19,7,94,Germany; Japan; United States,6,7.5,associative memory; dopamine neuron; visual le...,D. melanogaster,Neuroscience,None,None,2014


## 6. Quality checks

In [6]:
checks = pd.Series({
    "Rows match the number of matches the API reported": len(rows) == total,
    "No failed requests": not failed,
    "No duplicate articles": not df["article_id"].duplicated().any(),
    "Every article is tagged Neuroscience": df["subjects"].str.contains("Neuroscience").all(),
    f"Publication years fall between 2012 (eLife's launch) and {collection_date.year}":
        df["pub_year"].between(2012, collection_date.year).all(),
    "Every author list has at least one author": (df["n_authors"].dropna() >= 1).all(),
    "Every reference list has at least one reference": (df["n_references"].dropna() >= 1).all(),
    "Every affiliation belongs to an article in the data": affils["article_id"].isin(df["article_id"]).all(),
}, name="Passed")
checks.to_frame()

,Passed
Rows match the number of matches the API reported,True
No failed requests,True
No duplicate articles,True
Every article is tagged Neuroscience,True
Publication years fall between 2012 (eLife's launch) and 2026,True
Every author list has at least one author,True
Every reference list has at least one reference,True
Every affiliation belongs to an article in the data,True


## 7. Results

In [7]:
print(f"Observations: {len(df)} articles")
missing = pd.DataFrame({"Non-missing": df.notna().sum(), "Missing": df.isna().sum()})
missing["% missing"] = (100 * missing["Missing"] / len(df)).round(1)
missing

Observations: 480 articles


,Non-missing,Missing,% missing
article_id,480,0,0.0
title,480,0,0.0
article_type,480,0,0.0
published,480,0,0.0
n_authors,423,57,11.9
n_references,423,57,11.9
countries,423,57,11.9
n_funders,423,57,11.9
reference_age,423,57,11.9
keywords,423,57,11.9


The table below shows, for each article type, the percentage of API records that contain each field. It shows whether a missing value comes from the record itself or from my code.

In [8]:
(100 * pd.DataFrame(fields_present).groupby("article_type").mean()).round(1)

,authors,references,funding,keywords,researchOrganisms,elifeAssessment
article_type,,,,,,
research-article,100.0,100.0,100.0,100.0,88.7,27.4
reviewed-preprint,0.0,0.0,0.0,0.0,0.0,100.0


In [9]:
stats = df[["n_authors", "n_references", "n_funders", "reference_age", "pub_year"]].astype(float).agg(["count", "mean", "var"]).T
stats.columns = ["N", "Mean", "Variance"]
stats.round(2)

,N,Mean,Variance
n_authors,423.0,6.80,17.11
n_references,423.0,86.13,1113.99
n_funders,423.0,3.36,5.11
reference_age,423.0,9.31,8.98
pub_year,480.0,2021.67,10.09


## 8. Discussion

The scraper collected 480 articles on October 29, 2026, with no failed requests, and all of the quality checks passed. 423 are research articles and 57 are reviewed preprints. The missing values in the first table above all come from the reviewed preprints: the second table shows that their API records have no author list, reference list, funding list or keywords. Study organisms are also missing for 48 research articles, and the assessment ratings are missing for most articles because eLife only gives them under its review model from 2023 on. I don't drop anything here. The Project 2 report (`elife_neuromodulation_report.ipynb`) checks each of these gaps and the country names, and builds the final dataset from the two raw files.

The dataset has a few limitations. eLife matches "neuromodulation" anywhere in an article's full text, so some articles mention the term only in passing. Also, 2026 is a partial year, because the data stop at October 29.